## Setup

In [ ]:
import os
import sys
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langsmith import Client

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "context_hub_demo" else cwd
load_dotenv(repo_root / ".env", override=True)
sys.path.insert(0, str(repo_root))

if not os.getenv("LANGSMITH_API_KEY") and os.getenv("LANGSMITH_API_KEY_CORP"):
    os.environ["LANGSMITH_API_KEY"] = os.environ["LANGSMITH_API_KEY_CORP"]

assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY or LANGSMITH_API_KEY_CORP before running."

HUB_AGENT_NAME = os.getenv("NOVA_CONTEXT_HUB_AGENT", "nova")
HUB_WORKSPACE_ID = os.getenv("NOVA_CONTEXT_HUB_WORKSPACE", "4015447c-43ab-4414-8539-633d4cb47217")

os.environ["NOVA_CONTEXT_HUB_AGENT"] = HUB_AGENT_NAME
os.environ["NOVA_CONTEXT_HUB_WORKSPACE"] = HUB_WORKSPACE_ID
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "nova-context-hub-scene3")

print(f"Context Hub repo: {HUB_AGENT_NAME}")
print(f"Workspace: {HUB_WORKSPACE_ID}")
print(f"Tracing project: {os.environ['LANGSMITH_PROJECT']}")

## Add Agent Memory Locally

In [ ]:
AGENTS_MD = """# nova

Nova is a personal financial assistant. Help users understand spending, save more money, and make confident financial decisions.

## Subagents

You have no direct access to financial data. Delegate every financial query to the appropriate subagent:

- **spending_analyst**: spending patterns, category or merchant breakdowns, trends over time, and charts.
- **savings_advisor**: savings recommendations, what-if scenarios, and monthly or yearly projections.
- **account_manager**: account balances, recurring bills, and transfers to savings.

## Hard rules

- Always delegate financial questions to a subagent.
- If a chart is requested, preserve the chart data block returned by the subagent.
- No emojis.
- Be neutral and non-judgmental about spending.

## Applied skills

Apply these skills to every response:

- `currency-formatting`
- `chart-data-emission`
- `category-vocabulary`
"""

CURRENCY_FORMATTING_SKILL = """---
name: currency-formatting
description: How to format dollar amounts and percentages in Nova responses.
---

# currency-formatting

Use these rules whenever rendering money or percentages.

## Dollars

- Use a leading `$` and commas for thousands: `$1,234.56`.
- Show cents only when needed: `$45.30`, but `$200`.
- Negative balances use parentheses: `($45.00)`.

## Percentages

- Use one decimal place at most: `12.4%`.
- Drop trailing zeros: `15%`, not `15.0%`.
"""

CHART_DATA_EMISSION_SKILL = """---
name: chart-data-emission
description: How and when to emit chart data blocks for Nova's frontend.
---

# chart-data-emission

The frontend renders charts only when it finds a fenced `chartdata` block at the end of a response.

## When to emit

- The user asks for a chart, graph, pie, bar, line, area, or visualization.
- A subagent called `build_chart_spec` and returned chart JSON.

## Rules

- Put the fenced `chartdata` block last.
- Do not invent chart JSON.
- Never draw charts with ASCII art or unicode block characters.
"""

CATEGORY_VOCABULARY_SKILL = """---
name: category-vocabulary
description: Canonical spending-category vocabulary and common aliases.
---

# category-vocabulary

Use only these canonical categories in tool arguments, prose, and chart specs:

- `coffee`
- `fast_food`
- `delivery`
- `dining`
- `entertainment`
- `groceries`
- `transportation`
- `shopping`
- `subscription`
- `utilities`
- `healthcare`
- `income`
- `transfer`
- `other`

Aliases: restaurants -> dining, takeout -> delivery, gas -> transportation, streaming -> subscription, supermarket -> groceries.
"""

In [ ]:
from deepagents.backends.utils import create_file_data

STATE_FILES = {
    "/AGENTS.md": create_file_data(AGENTS_MD),
    "/skills/currency-formatting/SKILL.md": create_file_data(CURRENCY_FORMATTING_SKILL),
    "/skills/chart-data-emission/SKILL.md": create_file_data(CHART_DATA_EMISSION_SKILL),
    "/skills/category-vocabulary/SKILL.md": create_file_data(CATEGORY_VOCABULARY_SKILL),
}

pprint(sorted(STATE_FILES))

## Build the Agent


In [ ]:
from deepagents import SubAgent, create_deep_agent
from deepagents.backends import CompositeBackend, ContextHubBackend, StateBackend
from deepagents.middleware.skills import SkillsMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from src.prompts import (
    ACCOUNT_MANAGER_SYSTEM_PROMPT,
    SAVINGS_ADVISOR_SYSTEM_PROMPT,
    SKILLS_PROMPT_TEMPLATE,
    SPENDING_ANALYST_SYSTEM_PROMPT,
)
from src.tools.accounts import get_accounts, get_recurring_bills
from src.tools.charts import build_chart_spec
from src.tools.savings import calculate_savings_potential, get_savings_recommendation, transfer_to_savings
from src.tools.spending import get_category_spending, get_merchant_spending_pattern, get_spending_summary
from src.tools.transactions import get_recent_income, get_transactions

MODEL = os.getenv("MODEL", "anthropic:claude-haiku-4-5-20251001")


def build_nova_agent(
    backend,
    *,
    memory_path: str = "/AGENTS.md",
    skills_path: str = "/skills/",
):
    spending_analyst = SubAgent(
        name="spending_analyst",
        description="Deep dive into spending patterns, trends, categories, merchants, and charts.",
        model=MODEL,
        tools=[get_transactions, get_spending_summary, get_category_spending, get_merchant_spending_pattern, build_chart_spec],
        system_prompt=SPENDING_ANALYST_SYSTEM_PROMPT,
    )

    savings_advisor = SubAgent(
        name="savings_advisor",
        description="Calculate savings potential, what-if scenarios, and savings recommendations.",
        model=MODEL,
        tools=[get_transactions, get_recent_income, get_recurring_bills, get_savings_recommendation, calculate_savings_potential, build_chart_spec],
        system_prompt=SAVINGS_ADVISOR_SYSTEM_PROMPT,
    )

    account_manager = SubAgent(
        name="account_manager",
        description="Look up accounts, balances, recurring bills, and transfers.",
        model=MODEL,
        tools=[get_accounts, get_recurring_bills, transfer_to_savings],
        system_prompt=ACCOUNT_MANAGER_SYSTEM_PROMPT,
    )

    return create_deep_agent(
        model=MODEL,
        backend=backend,
        memory=[memory_path],
        middleware=[
            SkillsMiddleware(
                backend=backend,
                sources=[skills_path],
                system_prompt=SKILLS_PROMPT_TEMPLATE,
            )
        ],
        subagents=[spending_analyst, savings_advisor, account_manager],
        checkpointer=InMemorySaver(),
    )


state_backend = StateBackend()
state_agent = build_nova_agent(state_backend)

## Bring in Context Hub

In [ ]:
client = Client(workspace_id=HUB_WORKSPACE_ID)
context_hub_backend = ContextHubBackend(HUB_AGENT_NAME, client=client)

context_hub_backend

In [ ]:
def paths(ls_result):
    if ls_result.error:
        raise RuntimeError(ls_result.error)
    return [entry["path"] for entry in ls_result.entries]


def read_context_text(file_path: str, *, limit: int = 2000) -> str:
    result = context_hub_backend.read(file_path, limit=limit)
    if result.error:
        raise RuntimeError(result.error)
    return result.file_data["content"]


def markdown_context_file(file_path: str, *, limit: int = 2000):
    display(Markdown(f"### `{file_path}`\n\n" + read_context_text(file_path, limit=limit)))

In [ ]:
print("Context Hub root")
pprint(paths(context_hub_backend.ls("/")))

print("\nSkills in Context Hub")
pprint(paths(context_hub_backend.ls("/skills/")))

## Inspect the Context Hub Memory and Skills

In [ ]:
markdown_context_file("/AGENTS.md", limit=160)

for skill_path in [
    "/skills/currency-formatting/SKILL.md",
    "/skills/chart-data-emission/SKILL.md",
    "/skills/category-vocabulary/SKILL.md",
]:
    markdown_context_file(skill_path, limit=160)

## Swap the Backend

In [ ]:
# Before: everything lives in StateBackend state.
state_agent = build_nova_agent(StateBackend())

# Now create the durable backend: /memories/ is Context Hub; everything else is StateBackend scratchpad.
nova_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": context_hub_backend},
)

pprint({"default": "StateBackend scratchpad", "routes": {"/memories/": "ContextHubBackend"}})

# Now point memory and skills through the /memories/ mount.
context_agent = build_nova_agent(
    nova_backend,
    memory_path="/memories/AGENTS.md",
    skills_path="/memories/skills/",
)


## Tracing Configuration

In [ ]:
tracing_config = {
    "LANGSMITH_TRACING": os.environ["LANGSMITH_TRACING"],
    "LANGSMITH_PROJECT": os.environ["LANGSMITH_PROJECT"],
}

pprint(tracing_config)